In [2]:
#starting off and setting up
import pandas as pd
import numpy as np
from scipy.optimize import minimize

prices = pd.read_csv("prices.csv", index_col=0, parse_dates=True)
returns = prices.pct_change().dropna()
n_assets = len(returns.columns)

bounds = tuple((0, 1) for _ in range(n_assets))
initial = np.repeat(1/n_assets, n_assets)

In [3]:
#split the 10-year data into sample and testing ground
split_date = "2020-01-01"
sample = returns.loc[:split_date]
test=returns.loc[split_date:]
print(test.head())

                AAPL       CAT       JNJ       JPM        KO      MSFT  \
Date                                                                     
2020-01-02  0.022817  0.019299  0.000686  0.012124 -0.006504  0.018516   
2020-01-03 -0.009722 -0.013884 -0.011578 -0.013196 -0.005456 -0.012451   
2020-01-06  0.007968 -0.000674 -0.001248 -0.000795 -0.000366  0.002585   
2020-01-07 -0.004703 -0.013213  0.006107 -0.017001 -0.007682 -0.009118   
2020-01-08  0.016086  0.008881 -0.000138  0.007801  0.001843  0.015928   

                NVDA        PG       WMT       XOM  
Date                                                
2020-01-02  0.019592 -0.011930  0.000842  0.016050  
2020-01-03 -0.016006 -0.006725 -0.008828 -0.008039  
2020-01-06  0.004194  0.001387 -0.002036  0.007678  
2020-01-07  0.012106 -0.006192 -0.009265 -0.008184  
2020-01-08  0.001876  0.004263 -0.003432 -0.015080  


In [5]:
#get mean returns, covariances and set risk-free rate
sample_m = sample.mean() * 252
sample_cov = sample.cov() * 252
rfr = 0.02

#define function to get Sharpe ratio from given portfolio and maximise
def neg_sharpe_sample(w):
    r = np.sum(w * sample_m)
    v = np.sqrt(w.T @ sample_cov @ w)
    return -(r - rfr)/v

sum_to_one = {'type': 'eq', 'fun': lambda w: np.sum(w)-1}
sample_optimise = minimize(neg_sharpe_sample, initial, bounds=bounds, constraints=sum_to_one)
sample_w = sample_optimise.x

#get max Sharpe ratio and the portfolio producing it
print(-sample_optimise.fun)
print(pd.Series(sample_w, index=returns.columns).round(4))

1.497731739805984
AAPL    0.0325
CAT     0.0000
JNJ     0.0000
JPM     0.1612
KO      0.0849
MSFT    0.3068
NVDA    0.3196
PG      0.0775
WMT     0.0175
XOM     0.0000
dtype: float64


In [6]:
#now apply this optimised portfolio out-of-sample
oos_returns = test @ sample_w
print(oos_returns)
oos_ann_return = oos_returns.mean() *252
oos_ann_vol = oos_returns.std() * np.sqrt(252)
oos_sharpe = (oos_ann_return-rfr)/oos_ann_vol
print(oos_ann_return, oos_ann_vol, oos_sharpe)

Date
2020-01-02    0.013175
2020-01-03   -0.012518
2020-01-06    0.002305
2020-01-07   -0.003115
2020-01-08    0.007693
                ...   
2024-12-23    0.010915
2024-12-24    0.008617
2024-12-26   -0.000642
2024-12-27   -0.014376
2024-12-30   -0.006504
Length: 1257, dtype: float64
0.38506536974824274 0.29403873190011304 1.2415553807797604


In [9]:
#comapre against 1/n (naive)
equal_w = np.repeat(1/n_assets, n_assets)
ew_returns = test @ equal_w
ew_ann_return = ew_returns.mean() * 252
ew_ann_vol = ew_returns.std() * np.sqrt(252)
ew_sharpe = (ew_ann_return-rfr)/ew_ann_vol
print(ew_ann_return, ew_ann_vol, ew_sharpe)

0.2178936868660952 0.1922695699217547 1.0292512067646964


In [11]:
#now do oos testing across multiple timeframes
split_dates = ["2018-01-01", "2019-01-01", "2020-01-01", "2021-01-01", "2022-01-01"]
results=[]

def neg_sharpe_sample(w, mu, sig):
    r = np.sum(w * mu)
    v = np.sqrt(w.T @ sig @ w)
    return -(r - rfr)/v

for date in split_dates:
    sample=returns.loc[:date]
    test=returns.loc[date:]

    sample_m=sample.mean()*252
    sample_cov=sample.cov()*252
    sample_optimise = minimize(neg_sharpe_sample, initial, args=(sample_m, sample_cov), bounds=bounds, constraints=sum_to_one)
    opt_w = sample_optimise.x
    OOS_daily_rets = test @ opt_w
    OOS_sharpe = (OOS_daily_rets.mean()*252 - rfr) / (OOS_daily_rets.std()*np.sqrt(252))
    EW_daily_rets = test @ equal_w
    EW_sharpe = (EW_daily_rets.mean()*252 - rfr) / (EW_daily_rets.std()*np.sqrt(252))

    results.append({
    'Split': date,
    'In-sample Sharpe': -sample_optimise.fun,
    'OOS Sharpe': OOS_sharpe,
    'Equal-weighted Sharpe': EW_sharpe,
    'Max Weight': opt_w.max()
        })
DF_results=pd.DataFrame(results)
DF_results.round(3)

,Split,In-sample Sharpe,OOS Sharpe,Equal-weighted Sharpe,Max Weight
0,2018-01-01,2.119,1.125,1.029,0.494
1,2019-01-01,1.349,1.385,1.231,0.518
2,2020-01-01,1.498,1.242,1.086,0.320
3,2021-01-01,1.437,1.301,1.381,0.565
4,2022-01-01,1.528,1.057,1.145,0.549
